In [2]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import sklearn

In [ ]:
### Data

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 'massey_rank_dif', 'avg_margin_dif',
       'avg_eff_A', 'avg_opp_eff_A', 'avg_eff_B', 'avg_opp_eff_B',
       'avg_thr_per_A', 'ft_per_A', 'avg_fg_per_A', 'avg_fg_a_per_A',
       'avg_thr_a_per_A', 'avg_to_per_A', 'avg_blk_per_A',
       'avg_opp_fg_a_per_A', 'avg_opp_fg_per_A', 'avg_opp_to_per_A',
       'avg_thr_per_B', 'ft_per_B', 'avg_fg_per_B', 'avg_fg_a_per_B',
       'avg_thr_a_per_B', 'avg_to_per_B', 'avg_blk_per_B',
       'avg_opp_fg_a_per_B', 'avg_opp_fg_per_B', 'avg_opp_to_per_B',
       'thr_a_per_dif', 'tempo_pred', 'pred_fg_per_dif', 'pred_or_per_dif']

#first batch
training_base = training.query("Season <= 2015")
val_base = training.query("Season == 2016")

training_dfs = [training_base]
val_dfs = [val_base]

for i in range (2016, 2025):
    if i == 2020:
        i = i+1
        j = j+2
    elif i == 2019:
        j = i+2
    else:
        j = i+1

    temp_training = training.query("Season == @i")
    temp_val = training.query("Season == @j")

    training_dfs.append(temp_training)
    val_dfs.append(temp_val)

In [45]:
### Model

#model
mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, objective='binary:logistic')


for i in range(len(training_dfs)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = val_dfs[i]

    X_val = val[features]
    y_val = val['result']

    #fit model
    mod.fit(X_train, y_train)

    #make predictions
    preds = mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    print(np.mean(val['loss']))


#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

0.19953591836286794
0.23013416518994467
0.20600869093070154
0.17424205583340527
0.23010680465604239
0.2412415447347495
0.2636083334270156
0.23065123607176172
0.2033043746539437
0.17552365090341132
